# 03 - Models and backtests

Compare models on identical rolling-origin folds, look at where they win and lose, and
get prediction intervals for safety stock.

**The rule of this notebook:** nothing counts until it beats `ma28` (a 28-day trailing
mean) by more than its own cross-fold standard deviation. That baseline is much harder
than it sounds on intermittent retail demand.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from demandml import config as C
from demandml.experiment import RunConfig, run, compare_runs
from demandml.metrics import evaluate, evaluate_by, pinball, coverage
from demandml.models import list_models

pl.Config(tbl_rows=30, tbl_cols=15, tbl_width_chars=250)
sns.set_theme(style="whitegrid")

for n, note in list_models():
    print(f"{n:18s} {note}")

## A first leaderboard

`sku_limit` and `max_train_rows` keep this quick while iterating -- drop them for a real
number. Every model sees exactly the same rows, features and folds.

In [ ]:
cfg = RunConfig(
    models=["global_mean", "naive_last", "ma7", "ma28", "seasonal_blend", "croston",
            "ridge_log", "lgbm_tweedie", "lgbm_l2", "two_stage"],
    horizon=7,
    sku_limit=200,
    max_train_rows=250_000,
    splits=C.SplitSpec(test_size=28, n_folds=3),
)
res = run(cfg)
res.vs_baseline("ma28")

### Read this table properly

* **wape** -- total absolute error / total actual units. The headline.
* **wape_sd** -- spread across folds. *If a model's edge over the baseline is smaller
  than this, it has not beaten anything.*
* **bias** -- signed. Over- and under-forecasting cost very different amounts.
* **shortfall** -- unmet demand as a share of total: the stockout cost of using the
  forecast as-is.
* **mase** -- scaled by a naive in-sample MAE, so it is comparable across horizons.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
lb = res.leaderboard().filter(pl.col("model") != "global_mean")
ax[0].barh(lb.get_column("model"), lb.get_column("wape"),
           xerr=lb.get_column("wape_sd").fill_null(0), color="steelblue")
ax[0].invert_yaxis(); ax[0].set(title="WAPE (lower is better), +/- cross-fold sd", xlabel="wape")
ax[1].barh(lb.get_column("model"), lb.get_column("bias"), color="indianred")
ax[1].axvline(0, c="k", lw=1); ax[1].invert_yaxis()
ax[1].set(title="bias (>0 = over-forecast)", xlabel="bias")
plt.tight_layout()

## Accuracy decays with horizon

This curve is what decides your replenishment cadence and how much safety stock the
lead time needs.

In [ ]:
rows = []
for h in (1, 7, 14, 28):
    r = run(RunConfig(models=["ma28", "croston", "lgbm_tweedie"], horizon=h,
                      sku_limit=200, max_train_rows=250_000,
                      splits=C.SplitSpec(test_size=28, n_folds=2)), save=False)
    rows.append(r.leaderboard().with_columns(pl.lit(h).alias("h")))
sweep = pl.concat(rows)
display(sweep.select("h", "model", "wape", "bias", "mase").sort("h", "wape"))

fig, ax = plt.subplots(figsize=(8, 4.5))
for m in sweep.get_column("model").unique():
    d = sweep.filter(pl.col("model") == m).sort("h")
    ax.plot(d.get_column("h"), d.get_column("wape"), marker="o", label=m)
ax.set(xlabel="horizon (days)", ylabel="WAPE", title="error vs horizon"); ax.legend()

## Where does the model win?

Average WAPE is an average over very different SKUs. Break it down before believing it.
The usual finding: ML wins on high-volume smooth SKUs and loses to Croston on lumpy
ones -- which is an argument for routing by demand class, not for one global model.

In [ ]:
from demandml.profiling import demand_classes
from demandml.panel import load_panel

cls = demand_classes(load_panel("daily")).select(C.SKU, "demand_class", "total_units")
preds = res.predictions.join(cls, on=C.SKU, how="left")

by_class = (preds.group_by(["model", "demand_class"])
                 .agg((pl.col("y") - pl.col("yhat")).abs().sum().alias("abs_err"),
                      pl.col("y").sum().alias("actual"))
                 .with_columns((pl.col("abs_err") / pl.col("actual")).alias("wape"))
                 .pivot(on="demand_class", index="model", values="wape")
                 .sort("model"))
display(by_class)

In [ ]:
# Per-SKU: which series is the model actually bad at?
per_sku = res.by_group("lgbm_tweedie", "sku").join(cls, on="sku", how="left")
print("worst 10 SKUs by WAPE (min 30 test points):")
display(per_sku.sort("wape", descending=True).head(10))
print("best 10:")
display(per_sku.head(10))

## Actual vs predicted

In [ ]:
p = res.predictions.filter(pl.col("model") == "lgbm_tweedie")
top = (p.group_by(C.SKU).agg(pl.col("y").sum().alias("v"))
        .sort("v", descending=True).head(4).get_column(C.SKU).to_list())

fig, axes = plt.subplots(2, 2, figsize=(14, 6.5))
for sku, a in zip(top, axes.ravel()):
    d = p.filter(pl.col(C.SKU) == sku).sort(C.DATE)
    a.plot(d.get_column(C.DATE), d.get_column("y"), lw=1.2, label="actual")
    a.plot(d.get_column(C.DATE), d.get_column("yhat"), lw=1.2, ls="--", label="forecast")
    a.set_title(sku, fontsize=10); a.legend(fontsize=8)
plt.tight_layout()

In [ ]:
# Residual diagnostics: bias by weekday and by demand level.
r = p.with_columns((pl.col("yhat") - pl.col("y")).alias("err"),
                   pl.col(C.DATE).dt.weekday().alias("dow"))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
dow = r.group_by("dow").agg(pl.col("err").mean()).sort("dow")
ax[0].bar(dow.get_column("dow"), dow.get_column("err")); ax[0].axhline(0, c="k", lw=1)
ax[0].set(title="mean error by weekday (1=Mon)", ylabel="over-forecast")
buckets = r.with_columns(pl.col("y").cut([0.5, 2, 5, 10, 25, 50, 100]).alias("bucket"))
b = buckets.group_by("bucket").agg(pl.col("err").mean(), pl.len().alias("n")).sort("bucket")
ax[1].bar(range(len(b)), b.get_column("err")); ax[1].axhline(0, c="k", lw=1)
ax[1].set_xticks(range(len(b))); ax[1].set_xticklabels([str(x) for x in b.get_column("bucket")], rotation=45, ha="right")
ax[1].set(title="mean error by actual-demand bucket")
plt.tight_layout()

## Prediction intervals for safety stock

The mean forecast is not what a replenishment decision needs. For a 95% service level
you need roughly the P95 of demand over the lead time. `lgbm_quantile` fits one model
per quantile; check that the realised coverage matches the nominal level before you
trust it.

In [ ]:
from demandml.features import feature_columns, load_features
from demandml.splits import holdout, apply_fold
from demandml.models import get_model

lf = load_features("daily")
fts = feature_columns(lf)
ycol = "y_cum7"   # cumulative demand over the next 7 days -- the replenishment question

sub = (lf.filter(pl.col(ycol).is_not_null() & (pl.col("sample_weight") > 0))
         .select([C.SKU, C.DATE, ycol] + fts))
skus = sub.select(pl.col(C.SKU).unique().sort().head(150)).collect().get_column(C.SKU)
df = sub.filter(pl.col(C.SKU).is_in(skus.to_list())).collect()

fold = holdout(df.get_column(C.DATE).unique(), test_size=28, horizon=7)
print(fold.describe())
tr = apply_fold(df.lazy(), fold, "train").collect()
te = apply_fold(df.lazy(), fold, "test").collect()

qm = get_model("lgbm_quantile", n_estimators=400, num_leaves=63)
qm.fit(tr.select(fts), tr.get_column(ycol).to_numpy())
q = qm.predict_quantiles(te.select(fts))
y = te.get_column(ycol).to_numpy()

for col in q.columns:
    lvl = float(col[1:])
    print(f"{col}: pinball={pinball(y, q.get_column(col), lvl):.3f}  "
          f"share below={np.mean(y <= q.get_column(col).to_numpy()):.3f} (nominal {lvl})")
print(f"\nP10-P90 coverage: {coverage(y, q.get_column('q0.1'), q.get_column('q0.9')):.3f} (nominal 0.80)")

## Ideas worth testing next

The scaffolding is here; these are the experiments it exists to run.

1. **Relative target.** `ScaledTarget` divides units by the trailing 28-day mean so one
   model does not have to span three orders of magnitude of SKU volume. Usually the
   biggest single win on a multi-SKU panel.
2. **Tune the GBDT.** Defaults here are deliberately modest. `optuna` is installed;
   tune on folds 0-1 and report on fold 2 only.
3. **Route by demand class.** Croston for lumpy, GBDT for smooth, pick per SKU on a
   validation fold. Almost always beats any single global model.
4. **Predict `y_cum{lead_time}` directly** rather than summing daily forecasts -- fewer
   compounding errors, and it is the quantity the order decision needs.
5. **Handle censoring properly.** `drop_censored=True` throws information away. Compare
   against `units_uncensored` (the imputed target) and, if it matters, a Tobit model.
6. **Weekly panel.** `demandml panel --freq weekly` then `--freq weekly` features.
   Aggregation kills a lot of the intermittency noise; if the decision is weekly, model
   weekly.
7. **Global neural models.** With many series and years of daily history, DeepAR / N-BEATS /
   TFT are viable. Add them as a new `models/` module implementing `fit`/`predict`.